# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  
**Name:** Simi Chakravarty

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [10]:
q1 = q('''
SELECT t.artist_id, t.track_id, a.name, a.country
FROM tracks t
JOIN artists a
ON t.artist_id = a.artist_id
''')
q1

,artist_id,track_id,name,country
0,1,10,Nova Waves,US
1,1,11,Nova Waves,US
2,2,12,The Blue Ridge,US
3,3,13,Kestrel,UK
4,3,14,Kestrel,UK
5,4,15,Marisol,ES
6,2,16,The Blue Ridge,US
7,2,17,The Blue Ridge,US
8,3,18,Kestrel,UK


**In order to make a table that contained every track with its artist's name and country, I first selected the necessary columns from each table, which would be artist_id, track_id, name, and country, and specified that I would need them from either the tracks (t) or artists (a) tables. Since artist_id is the common column between the tracks and artists tables, I used JOIN, which is a type of inner join, to join these tables together with the specified columns using artist_id (ON t.artist_id = a.artist_id).**

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT t.genre, AVG(t.seconds)
FROM tracks t
GROUP BY t.genre
ORDER BY AVG(t.seconds) DESC
LIMIT 1
''')

,genre,AVG(t.seconds)
0,Electronic,287.5


**To return only the genre and average track length, I first needed to select the genre column from the tracks table and then create a new column with the average track length using AVG on the seconds column in the tracks table. Next, I grouped all of the songs by genre using GROUP BY and ordered them in a descending fashion so that the genre with the longest average track length would show up at the top. Finally, in order to make it so that only the genre with the longest average track length was shown, I used LIMIT 1.**

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT p.user, COUNT(p.play_id) AS plays, COUNT(DISTINCT p.track_id) AS tracks
FROM plays p
GROUP BY p.user
''')
q3

,user,plays,tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


**The first thing that I did was select the user column from the plays table, using COUNT to count each play_id as a play and each track_id as a track. I then grouped the plays and tracks by user to create a table that distinguished the number of plays and tracks that each user had.**

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [24]:
q4 = q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p
ON t.track_id = p.track_id
WHERE p.play_id IS NULL
''')
q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


**In order to find the tracks which have never been played, I selected the track_id and title from the tracks table and used a LEFT JOIN to connect it to the plays table, which has the shared track_id column. I used a left join because I wanted to make sure that the tracks which had never been played (the ones with no play_id) would still be included when the tables were joined. To select solely the tracks which haven't been played, I filtered the table using WHERE, specifically when p.play_id IS NULL.**

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [29]:
q('''
SELECT a.name, SUM(t.seconds) AS total_seconds, ROUND(SUM(t.seconds) / 60, 1) AS total_minutes
FROM artists a
JOIN tracks t
ON a.artist_id = t.artist_id
JOIN plays p
ON t.track_id = p.track_id
GROUP BY a.name
ORDER BY SUM(t.seconds) DESC
''')

,name,total_seconds,total_minutes
0,Kestrel,1175,19.0
1,Nova Waves,843,14.0
2,The Blue Ridge,384,6.0
3,Marisol,210,3.0


**The first think that I did was SELECT the artist name to be included in the table, then used SUM to determine the total seconds of music each artist had, and finally used SUM of t.seconds and divided it by 30 to get the total number of minutes. In order to make sure that the number of seconds was accurate, I had to join the artist table to the tracks table and then chain the join to the plays table. This is to account for tracks that are played multiple times produced by the same artist. Finally, I used GROUP to make sure the seconds and minutes were condensed for the same artists and then used ORDER BY to sort the seconds and minutes from highest to lowest.**

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT t.track_id, t.title
FROM tracks t
WHERE t.genre IS NULL
''')

,track_id,title
0,18,Untitled Demo


**I first selected the track_id and title from the tracks table to make sure that these columns were returned in the table. To return only the tracks without a genre, I used WHERE to filter the table only when t.genre IS NULL, which returns the tracks that are missing a genre. If I had used, WHERE genre != 'Pop', it would have returned all of the genres that weren't Pop and their respective titles because != means not equal to.**

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT p.played_on, COUNT(p.play_id) AS plays, COUNT(DISTINCT p.user) AS users
FROM plays p
GROUP BY p.played_on
ORDER BY p.played_on ASC
''')

,played_on,plays,users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


**In order to create a table with the plays per day, I first selected the played_on column from the plays table and created two new columns: plays which uses the number of instances of play_id and users which uses the number of instances of each user. Then, I used GROUP BY to make sure that plays and users were sorted by the unqiue date that each user played. To make sure that the earliest date was listed first, I had to use ORDER BY and make sure the order was ascending.**

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


**To make sure that these assertions ran, I had to go back to some of my previous questions and set them equal to q1, q3, and q4 respectively.**

### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

**The query which gave me the most trouble was Query 4, where I had to list the tracks which had never been played. I was having trouble with this question because I was initially using an inner JOIN, which resulted in all of the XXX.**